# 电商用户行为分析完整项目

基于阿里天池移动端脱敏用户行为数据，完成电商用户行为分析，包含四大模块：

| 模块 | 内容 |
|------|------|
| 模块1 | EDA 探索性数据分析 |
| 模块2 | 电商转化漏斗分析 |
| 模块3 | RFM 用户分层分析 |
| 模块4 | AB-Test 仿真实验 |

**数据来源**：阿里天池「阿里移动推荐算法」数据集
**时间范围**：2014-11-18 ~ 2014-12-18

## 0. 环境配置与数据加载

导入所需库，配置中文字体防止图表乱码，加载并预处理数据。

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ---------- 第一步：先设置 seaborn 风格（seaborn 会重置 rcParams，必须放在字体设置之前）----------
sns.set_style("whitegrid")
sns.set_palette("Set2")

# ---------- 第二步：配置中文字体（防止乱码）----------
# 1. 重建 matplotlib 字体缓存，确保所有系统字体可被识别
fm._load_fontmanager(try_read_cache=False)

# 2. 按优先级排列中文字体
_chinese_font_priority = [
    "Microsoft YaHei", "Microsoft YaHei UI", "SimHei",
    "Noto Sans SC", "SimSun", "DengXian", "KaiTi",
    "FangSong", "Arial Unicode MS", "DejaVu Sans",
]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_chinese_fonts_available = [f for f in _chinese_font_priority if f in _available_fonts]

# 3. 设置全局字体（必须在 seaborn set_style 之后，否则会被覆盖）
matplotlib.rcParams["font.sans-serif"] = _chinese_fonts_available
matplotlib.rcParams["axes.unicode_minus"] = False

# 4. seaborn 也设置字体（双重保险）
sns.set_context("notebook", font_scale=1.0, rc={"font.sans-serif": _chinese_fonts_available})

print(f"[字体配置] 使用中文字体: {_chinese_fonts_available[:3]}")

# 图片输出目录
IMG_DIR = "img"
os.makedirs(IMG_DIR, exist_ok=True)

# 数据文件路径
DATA_PATH = "data/tianchi_mobile_recommend_train_user.csv"

# 行为类型映射
BEHAVIOR_MAP = {1: "浏览", 2: "收藏", 3: "加购", 4: "购买"}
BEHAVIOR_COLORS = {"浏览": "#4C72B0", "收藏": "#55A868", "加购": "#DD8452", "购买": "#C44E52"}

In [ ]:
def load_data():
    """加载并预处理数据"""
    df = pd.read_csv(
        DATA_PATH,
        dtype={
            "user_id": "int32", "item_id": "int32",
            "behavior_type": "int8", "item_category": "int32",
        },
    )
    if "user_geohash" in df.columns:
        df.drop("user_geohash", axis=1, inplace=True)
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    if "behavior_name" not in df.columns:
        df["behavior_name"] = df["behavior_type"].map(BEHAVIOR_MAP)
    df["date"] = df["time"].dt.date
    df["hour"] = df["time"].dt.hour
    df["day_of_week"] = df["time"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6])
    df = df.dropna(subset=["time"])
    print(f"总记录数：{len(df):,}")
    print(f"独立用户数：{df['user_id'].nunique():,}")
    print(f"独立商品数：{df['item_id'].nunique():,}")
    print(f"独立类目数：{df['item_category'].nunique():,}")
    print(f"时间范围：{df['time'].min()} ~ {df['time'].max()}")
    return df

df = load_data()

## 模块1：EDA 探索性数据分析

In [ ]:
# --- 1.1 基础统计 ---
total_records = len(df)
unique_users = df["user_id"].nunique()
unique_items = df["item_id"].nunique()
unique_categories = df["item_category"].nunique()

behavior_counts = df["behavior_type"].value_counts().sort_index()
behavior_ratios = (behavior_counts / total_records * 100).round(2)

print(f"总行为记录数：{total_records:,}")
print(f"独立用户数：{unique_users:,}")
print(f"独立商品数：{unique_items:,}")
print(f"独立类目数：{unique_categories:,}")
for btype in sorted(BEHAVIOR_MAP.keys()):
    name = BEHAVIOR_MAP[btype]
    print(f"  {name}：{behavior_counts.get(btype, 0):,}（{behavior_ratios.get(btype, 0)}%）")

In [ ]:
# 图1：行为类型分布（柱状图 + 环形饼图）
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
names = [BEHAVIOR_MAP[i] for i in behavior_counts.index]
colors = [BEHAVIOR_COLORS[n] for n in names]
values = behavior_counts.values
total_val = values.sum()

# 左图：柱状图（y轴以"万"为单位）
bars = axes[0].bar(names, values, color=colors, edgecolor="white", linewidth=0.8, width=0.6)
axes[0].set_title("各类行为数量分布", fontsize=14, fontweight="bold", pad=10)
axes[0].set_ylabel("记录数（万次）", fontsize=12)
axes[0].yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/10000:.0f}")
)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{val/10000:.1f}万", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_ylim(0, max(values) * 1.15)

# 右图：环形饼图 + 外部图例（避免小扇区标签重叠）
wedges, _ = axes[1].pie(
    values, colors=colors, startangle=90,
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
)
legend_labels = [
    f"{name}  {val/total_val*100:.2f}%  ({val:,})"
    for name, val in zip(names, values)
]
axes[1].legend(wedges, legend_labels, loc="center left",
               bbox_to_anchor=(1.02, 0.5), fontsize=11, frameon=False,
               title="行为类型  占比  数量", title_fontsize=11)
axes[1].set_title("各类行为占比", fontsize=14, fontweight="bold", pad=10)
axes[1].text(0, 0, f"总计\n{total_val/10000:.0f}万",
             ha="center", va="center", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/01_behavior_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 1.2 时间维度：日UV与日行为总量趋势 ---
daily_stats = df.groupby("date").agg(
    daily_uv=("user_id", "nunique"),
    daily_behavior=("user_id", "count"),
).reset_index()

# 计算第几天（相对第一天）
start_date = daily_stats["date"].iloc[0]
daily_stats["day_num"] = [(d - start_date).days + 1 for d in daily_stats["date"]]

fig, ax1 = plt.subplots(figsize=(14, 5.5))
line1 = ax1.plot(daily_stats["day_num"], daily_stats["daily_uv"], color="#4C72B0",
                 marker="o", markersize=4, linewidth=1.8, label="日UV（独立用户数）")
ax1.set_xlabel("日期", fontsize=12)
ax1.set_ylabel("日UV（人）", fontsize=12, color="#4C72B0")
ax1.tick_params(axis="y", labelcolor="#4C72B0", labelsize=10)
ax1.tick_params(axis="x", labelsize=10)
ax1.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/1000:.0f}k")
)

# 左轴范围：让 UV 线整体偏低
uv_min, uv_max = daily_stats["daily_uv"].min(), daily_stats["daily_uv"].max()
ax1.set_ylim(uv_min * 0.9, uv_max * 1.15)

ax2 = ax1.twinx()
line2 = ax2.plot(daily_stats["day_num"], daily_stats["daily_behavior"], color="#C44E52",
                 marker="s", markersize=4, linewidth=1.8, label="日行为总量")
ax2.set_ylabel("日行为总量（万次）", fontsize=12, color="#C44E52")
ax2.tick_params(axis="y", labelcolor="#C44E52", labelsize=10)
ax2.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/10000:.0f}")
)

# 右轴范围：让行为总量线始终在 UV 线上方
beh_min, beh_max = daily_stats["daily_behavior"].min(), daily_stats["daily_behavior"].max()
ax2.set_ylim(0, beh_max / 0.78)

# 统一 x 轴刻度：每 5 天一个标签
tick_step = 5
tick_positions = list(range(1, len(daily_stats) + 1, tick_step))
if len(daily_stats) not in tick_positions:
    tick_positions.append(len(daily_stats))
ax1.set_xticks(tick_positions)
ax1.set_xticklabels([f"第{d}天" for d in tick_positions], fontsize=10)

plt.title("日UV与日行为总量趋势（2014-11-18 ~ 2014-12-18）",
          fontsize=14, fontweight="bold", pad=12)
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left", fontsize=11, framealpha=0.9)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/02_daily_trend.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 工作日 vs 周末日均行为对比
weekday_df = df[~df["is_weekend"]]
weekend_df = df[df["is_weekend"]]
weekday_days = weekday_df["date"].nunique()
weekend_days = weekend_df["date"].nunique()
weekday_avg = weekday_df["behavior_type"].value_counts().sort_index() / weekday_days
weekend_avg = weekend_df["behavior_type"].value_counts().sort_index() / weekend_days

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(4)
width = 0.32
bar_names = [BEHAVIOR_MAP[i] for i in range(1, 5)]
bars1 = ax.bar(x - width / 2, weekday_avg.values, width,
               label=f"工作日（{weekday_days}天日均）", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width / 2, weekend_avg.values, width,
               label=f"周末（{weekend_days}天日均）", color="#C44E52", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(bar_names, fontsize=12)
ax.set_title("工作日 vs 周末日均行为对比", fontsize=14, fontweight="bold", pad=10)
ax.set_ylabel("日均行为次数（次）", fontsize=12)
ax.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/1000:.1f}k" if x >= 1000 else f"{x:.0f}")
)
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h,
                f"{h:,.0f}", ha="center", va="bottom", fontsize=9.5)
ax.legend(fontsize=11, loc="upper right")
ax.set_ylim(0, max(weekday_avg.max(), weekend_avg.max()) * 1.18)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/03_weekday_weekend.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 24小时行为分布（双行子图，浏览单独展示，其他行为放大展示）
hourly_behavior = df.groupby(["hour", "behavior_type"]).size().unstack(fill_value=0)

fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(13, 10), sharex=True,
                                        gridspec_kw={"height_ratios": [2.5, 1.5]})

# 上图：浏览
ax_top.plot(hourly_behavior.index, hourly_behavior[1],
            marker="o", markersize=4, linewidth=2,
            color=BEHAVIOR_COLORS["浏览"], label="浏览")
ax_top.set_ylabel("浏览次数", fontsize=12)
ax_top.set_title("24小时行为分布", fontsize=14, fontweight="bold", pad=10)
ax_top.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/10000:.0f}万" if x >= 10000 else f"{x:,.0f}")
)
ax_top.legend(fontsize=11, loc="upper left")
ax_top.grid(True, alpha=0.3)

# 下图：收藏、加购、购买
for btype in [2, 3, 4]:
    name = BEHAVIOR_MAP[btype]
    ax_bottom.plot(hourly_behavior.index, hourly_behavior[btype],
                   marker="o", markersize=4, linewidth=2,
                   color=BEHAVIOR_COLORS[name], label=name)
ax_bottom.set_xlabel("小时", fontsize=12)
ax_bottom.set_ylabel("行为次数", fontsize=12)
ax_bottom.set_xticks(range(0, 24))
ax_bottom.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x:,.0f}")
)
ax_bottom.legend(fontsize=11, loc="upper left")
ax_bottom.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/04_hourly_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# --- 1.3 类目分析 ---
# Top 20 类目总行为数
top_categories = df["item_category"].value_counts().head(20)
top_purchase_categories = df[df["behavior_type"] == 4]["item_category"].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左：Top 20 类目行为分布
top_cat_sorted = top_categories.sort_values()
axes[0].barh(range(len(top_cat_sorted)), top_cat_sorted.values,
             color="#4C72B0", edgecolor="white", height=0.7)
axes[0].set_yticks(range(len(top_cat_sorted)))
axes[0].set_yticklabels(top_cat_sorted.index, fontsize=10)
axes[0].set_title("Top 20 商品类目行为分布", fontsize=14, fontweight="bold", pad=10)
axes[0].set_xlabel("行为记录数", fontsize=12)
axes[0].set_ylabel("类目ID", fontsize=12)
axes[0].xaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/10000:.0f}万")
)

# 右：Top 20 类目购买分布
top_pur_sorted = top_purchase_categories.sort_values()
axes[1].barh(range(len(top_pur_sorted)), top_pur_sorted.values,
             color="#C44E52", edgecolor="white", height=0.7)
axes[1].set_yticks(range(len(top_pur_sorted)))
axes[1].set_yticklabels(top_pur_sorted.index, fontsize=10)
axes[1].set_title("Top 20 商品类目购买次数分布", fontsize=14, fontweight="bold", pad=10)
axes[1].set_xlabel("购买次数", fontsize=12)
axes[1].set_ylabel("类目ID", fontsize=12)
axes[1].xaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x:,.0f}")
)

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/05_06_top_categories.png", dpi=150, bbox_inches="tight")
plt.show()

# 分别保存单图
fig1, ax1 = plt.subplots(figsize=(12, 7))
ax1.barh(range(len(top_cat_sorted)), top_cat_sorted.values,
         color="#4C72B0", edgecolor="white", height=0.7)
ax1.set_yticks(range(len(top_cat_sorted)))
ax1.set_yticklabels(top_cat_sorted.index, fontsize=10)
ax1.set_title("Top 20 商品类目行为分布", fontsize=14, fontweight="bold", pad=10)
ax1.set_xlabel("行为记录数", fontsize=12)
ax1.set_ylabel("类目ID", fontsize=12)
ax1.xaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x/10000:.0f}万")
)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/05_top_categories.png", dpi=150, bbox_inches="tight")
plt.close()

fig2, ax2 = plt.subplots(figsize=(12, 7))
ax2.barh(range(len(top_pur_sorted)), top_pur_sorted.values,
         color="#C44E52", edgecolor="white", height=0.7)
ax2.set_yticks(range(len(top_pur_sorted)))
ax2.set_yticklabels(top_pur_sorted.index, fontsize=10)
ax2.set_title("Top 20 商品类目购买次数分布", fontsize=14, fontweight="bold", pad=10)
ax2.set_xlabel("购买次数", fontsize=12)
ax2.set_ylabel("类目ID", fontsize=12)
ax2.xaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x:,.0f}")
)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/06_top_purchase_categories.png", dpi=150, bbox_inches="tight")
plt.close()
print("[图表已保存] 05_top_categories.png, 06_top_purchase_categories.png")

## 模块2：电商转化漏斗分析

**业务路径**：浏览 → 收藏 → 加购 → 购买

> ⚠️ **重要口径**：漏斗每一步统计**去重独立 user_id（用户粒度）**，不统计行为记录条数。

In [ ]:
browse_users = set(df[df["behavior_type"] == 1]["user_id"].unique())
fav_users = set(df[df["behavior_type"] == 2]["user_id"].unique())
cart_users = set(df[df["behavior_type"] == 3]["user_id"].unique())
buy_users = set(df[df["behavior_type"] == 4]["user_id"].unique())

funnel_steps = [
    ("浏览", len(browse_users)),
    ("收藏", len(browse_users & fav_users)),
    ("加购", len(browse_users & fav_users & cart_users)),
    ("购买", len(browse_users & fav_users & cart_users & buy_users)),
]

funnel_df = pd.DataFrame(funnel_steps, columns=["步骤", "用户数"])
funnel_df["步骤转化率"] = (funnel_df["用户数"] / funnel_df["用户数"].shift(1)).fillna(1.0)
funnel_df["步骤转化率"] = (funnel_df["步骤转化率"] * 100).round(2).astype(str) + "%"
funnel_df["流失率"] = ((1 - funnel_df["用户数"] / funnel_df["用户数"].shift(1)) * 100).round(2)
funnel_df["流失率"] = funnel_df["流失率"].fillna(0).astype(str) + "%"
funnel_df["总体转化率"] = (funnel_df["用户数"] / funnel_df["用户数"].iloc[0] * 100).round(2).astype(str) + "%"

print("漏斗统计（用户粒度去重）：")
print(funnel_df.to_string(index=False))
print(f"\n浏览→购买整体转化率：{funnel_steps[-1][1] / funnel_steps[0][1]*100:.2f}%")

In [ ]:
# 漏斗图：标准倒三角形漏斗（顶宽=当前步，底宽=下一步，逐层递减）
fig, ax = plt.subplots(figsize=(12, 6))

step_values = [s[1] for s in funnel_steps]
max_val = step_values[0]
funnel_colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
widths = [v / max_val for v in step_values]  # 归一化宽度

for i, (name, val) in enumerate(funnel_steps):
    y_bottom = len(funnel_steps) - i - 1
    y_top = y_bottom + 1

    # 顶宽 = 当前步的归一化宽度
    top_w = widths[i]
    top_left = (1 - top_w) / 2
    # 底宽 = 下一步的归一化宽度（最后一层略微收窄，形成尖底）
    bottom_w = widths[i + 1] if i < len(funnel_steps) - 1 else top_w * 0.88
    bottom_left = (1 - bottom_w) / 2

    # 绘制梯形：顶宽=当前步，底宽=下一步（逐层递减）
    x = [top_left, 1 - top_left, 1 - bottom_left, bottom_left]
    y = [y_top, y_top, y_bottom, y_bottom]
    ax.fill(x, y, color=funnel_colors[i], alpha=0.85, edgecolor="white", linewidth=2)

    # 层内标注：行为名称 + 用户数
    ax.text(0.5, y_bottom + 0.5, f"{name}\n{val:,}人",
            ha="center", va="center", fontsize=13, fontweight="bold", color="white")

    # 右侧标注：步骤转化率与流失率
    if i > 0:
        prev_val = funnel_steps[i - 1][1]
        step_cr = val / prev_val * 100
        ax.text(1.02, y_bottom + 0.5,
                f"步骤转化率: {step_cr:.1f}%\n流失率: {100 - step_cr:.1f}%",
                ha="left", va="center", fontsize=10, color="#333333",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#F0F0F0", alpha=0.8))

ax.set_xlim(-0.08, 1.35)
ax.set_ylim(-0.1, len(funnel_steps) + 0.1)
ax.set_title("电商转化漏斗（用户粒度去重）", fontsize=15, fontweight="bold", pad=15)
ax.axis("off")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/07_funnel_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 模块3：RFM 用户分层分析

> ⚠️ **数据集没有消费金额 M（Monetary）**，使用用户购买频次替代 M 指标。

In [ ]:
data_end_date = pd.Timestamp("2014-12-18")
buy_df = df[df["behavior_type"] == 4].copy()
all_users = df["user_id"].unique()

rfm = buy_df.groupby("user_id").agg(
    last_purchase_time=("time", "max"),
    F=("user_id", "count"),
).reset_index()
rfm["R_days"] = (data_end_date - rfm["last_purchase_time"]).dt.days
rfm["M"] = rfm["F"]  # M 指标复用购买频次

rfm["R_score"] = pd.qcut(rfm["R_days"], q=5, labels=[5, 4, 3, 2, 1], duplicates="drop").astype(int)
rfm["F_score"] = pd.qcut(rfm["F"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = rfm["F_score"]

r_mean = rfm["R_score"].mean()
f_mean = rfm["F_score"].mean()

def classify(row):
    r, f = row["R_score"], row["F_score"]
    if r >= r_mean and f >= f_mean:
        return "高价值用户"
    elif r >= r_mean and f < f_mean:
        return "潜力用户"
    elif r < r_mean and f >= f_mean:
        return "一般用户"
    else:
        return "流失用户"

rfm["用户分层"] = rfm.apply(classify, axis=1)
no_purchase_count = len(all_users) - len(rfm)
segment_counts = rfm["用户分层"].value_counts()
segment_counts["无购买用户"] = no_purchase_count
total_users = len(all_users)

segment_df = pd.DataFrame({
    "用户数": segment_counts,
    "占比": (segment_counts / total_users * 100).round(2),
})
print(segment_df.to_string())

In [ ]:
# 图8：用户分层分布（环形饼图 + 柱状图）
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
seg_order = ["高价值用户", "潜力用户", "一般用户", "流失用户", "无购买用户"]
seg_colors = ["#C44E52", "#DD8452", "#4C72B0", "#999999", "#CCCCCC"]
seg_values = [segment_counts.get(s, 0) for s in seg_order]
seg_c = [c for c, v in zip(seg_colors, seg_values) if v > 0]
seg_l = [s for s, v in zip(seg_order, seg_values) if v > 0]
seg_v = [v for v in seg_values if v > 0]
total_seg = sum(seg_v)

# 环形饼图 + 外部图例
wedges, _ = axes[0].pie(
    seg_v, colors=seg_c, startangle=90,
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
)
legend_labels = [
    f"{label}  {val/total_seg*100:.1f}%  ({val:,}人)"
    for label, val in zip(seg_l, seg_v)
]
axes[0].legend(wedges, legend_labels, loc="center left",
               bbox_to_anchor=(1.02, 0.5), fontsize=10.5, frameon=False,
               title="用户分层  占比  人数", title_fontsize=11)
axes[0].set_title("RFM用户分层占比", fontsize=14, fontweight="bold", pad=10)
axes[0].text(0, 0, f"总计\n{total_seg:,}人",
             ha="center", va="center", fontsize=14, fontweight="bold")

# 柱状图
bars = axes[1].bar(seg_l, seg_v, color=seg_c, edgecolor="white", width=0.6)
axes[1].set_title("RFM用户分层数量", fontsize=14, fontweight="bold", pad=10)
axes[1].set_ylabel("用户数（人）", fontsize=12)
for bar, val in zip(bars, seg_v):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{val:,}", ha="center", va="bottom", fontsize=10.5, fontweight="bold")
axes[1].tick_params(axis="x", rotation=25, labelsize=10.5)
axes[1].set_ylim(0, max(seg_v) * 1.18)
axes[1].yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, p: f"{x:,.0f}")
)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/08_rfm_segment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 图9：R vs F 散点图（按分层着色，极端值截断标注）
fig, ax = plt.subplots(figsize=(11, 7))
scatter_colors = {
    "高价值用户": "#C44E52", "潜力用户": "#DD8452",
    "一般用户": "#4C72B0", "流失用户": "#999999",
}

# y 轴截断阈值：使用 95 分位数，避免极端值压缩主体分布
f_upper = int(rfm["F"].quantile(0.95))
f_upper = max(f_upper, 50)
outlier_count = 0

for seg, color in scatter_colors.items():
    subset = rfm[rfm["用户分层"] == seg]
    if len(subset) > 0:
        # 正常范围内的点
        normal = subset[subset["F"] <= f_upper]
        if len(normal) > 0:
            np.random.seed(42)
            jitter_r = np.random.uniform(-0.4, 0.4, len(normal))
            jitter_f = np.random.uniform(-0.3, 0.3, len(normal))
            ax.scatter(normal["R_days"] + jitter_r, normal["F"] + jitter_f,
                       c=color, label=seg, alpha=0.5, s=12, edgecolors="none")

        # 极端值点：画在上边界，用三角标记
        outliers = subset[subset["F"] > f_upper]
        if len(outliers) > 0:
            outlier_count += len(outliers)
            np.random.seed(43)
            jitter_r = np.random.uniform(-0.4, 0.4, len(outliers))
            ax.scatter(outliers["R_days"] + jitter_r, [f_upper] * len(outliers),
                       c=color, marker="^", s=60, edgecolors="white", linewidth=0.5,
                       alpha=0.8, zorder=5)

# 画上边界截断线，并标注极端值信息
ax.axhline(y=f_upper, color="#666666", linestyle="--", linewidth=1.2, alpha=0.7)
ax.text(1, f_upper * 1.03,
        f"▲ F>{f_upper} 的极端值（共{outlier_count}个）",
        ha="left", va="bottom", fontsize=10, color="#666666")

ax.set_xlabel("R（最近购买距今天数）", fontsize=12)
ax.set_ylabel("F（购买频次，次）", fontsize=12)
ax.set_title("RFM 用户 R-F 散点分布", fontsize=14, fontweight="bold", pad=10)
ax.legend(fontsize=11, loc="upper right", markerscale=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, f_upper * 1.15)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/09_rfm_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


## 模块4：AB-Test 仿真实验

> **⚠️ 本数据集没有真实线上 AB 分组，这是基于历史用户的仿真实验，仅用于演示完整 AB 测试流程，不是真实业务实验结果。**

In [ ]:
# 4.1 随机分流
all_users_arr = df["user_id"].unique()
np.random.seed(42)
np.random.shuffle(all_users_arr)
split_point = len(all_users_arr) // 2
control_users = set(all_users_arr[:split_point])
treatment_users = set(all_users_arr[split_point:])

n_control = len(control_users)
n_treatment = len(treatment_users)
n_total = n_control + n_treatment

print(f"总用户数：{n_total:,}")
print(f"对照组：{n_control:,}（{n_control/n_total*100:.1f}%）")
print(f"实验组：{n_treatment:,}（{n_treatment/n_total*100:.1f}%）")

# 4.2 SRM 校验
observed = np.array([n_control, n_treatment])
expected = np.array([n_total / 2, n_total / 2])
chi2_stat = ((observed - expected) ** 2 / expected).sum()
p_srm = 1 - stats.chi2.cdf(chi2_stat, df=1)
print(f"\nSRM卡方统计量：{chi2_stat:.4f}，p值：{p_srm:.4f}")
print("结论：分流比例符合50%:50%，无SRM问题。" if p_srm > 0.05 else "结论：存在SRM问题。")

In [ ]:
# 4.3 仿真模拟 + 4.4 转化率计算
buy_users_set = set(df[df["behavior_type"] == 4]["user_id"].unique())
control_buyers = control_users & buy_users_set
treatment_buyers_real = treatment_users & buy_users_set
control_cr_base = len(control_buyers) / n_control

target_lift = 0.03
target_treatment_cr = control_cr_base * (1 + target_lift)
target_treatment_buyers = int(target_treatment_cr * n_treatment)
additional_needed = max(0, target_treatment_buyers - len(treatment_buyers_real))

treatment_non_buyers = list(treatment_users - buy_users_set)
np.random.seed(123)
simulated_new_buyers = set(np.random.choice(treatment_non_buyers, additional_needed, replace=False)) if additional_needed > 0 else set()
treatment_buyers_simulated = treatment_buyers_real | simulated_new_buyers

control_cr = len(control_buyers) / n_control
treatment_cr = len(treatment_buyers_simulated) / n_treatment
abs_diff = treatment_cr - control_cr
rel_lift = (treatment_cr - control_cr) / control_cr

print(f"对照组转化率：{control_cr*100:.4f}%（{len(control_buyers):,}/{n_control:,}）")
print(f"实验组转化率：{treatment_cr*100:.4f}%（{len(treatment_buyers_simulated):,}/{n_treatment:,}）")
print(f"绝对差异：{abs_diff*100:.4f}个百分点")
print(f"相对提升：{rel_lift*100:.2f}%")

In [ ]:
# 4.5 两样本比例Z检验
x_c = len(control_buyers)
x_t = len(treatment_buyers_simulated)
p_pool = (x_c + x_t) / (n_control + n_treatment)
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
z_stat = (treatment_cr - control_cr) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
z_alpha_2 = 1.96
ci_lower = abs_diff - z_alpha_2 * se
ci_upper = abs_diff + z_alpha_2 * se

print(f"Z统计量：{z_stat:.4f}")
print(f"p值：{p_value:.6f}")
print(f"95%置信区间：[{ci_lower*100:.4f}%, {ci_upper*100:.4f}%]")
print("结论：p < 0.05，差异统计显著。" if p_value < 0.05 else "结论：差异不显著。")

In [ ]:
# 图10：AB-Test转化率对比
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
groups = ["对照组\n(control)", "实验组\n(treatment)"]
crs = [control_cr * 100, treatment_cr * 100]
colors_ab = ["#4C72B0", "#C44E52"]
bars = axes[0].bar(groups, crs, color=colors_ab, edgecolor="white", width=0.5)
axes[0].set_title("AB-Test 购买转化率对比", fontsize=14, fontweight="bold", pad=10)
axes[0].set_ylabel("转化率 (%)", fontsize=12)
axes[0].set_ylim(0, max(crs) * 1.25)
for bar, val in zip(bars, crs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{val:.2f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[0].text(0.5, max(crs) * 1.1,
             f"lift = +{rel_lift*100:.2f}%",
             ha="center", va="bottom", fontsize=13, color="#C44E52", fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#FFF0F0", edgecolor="#C44E52"))

ci_low_pct = ci_lower * 100
ci_high_pct = ci_upper * 100
diff_pct = abs_diff * 100
yerr_lower = [[diff_pct - ci_low_pct], [ci_high_pct - diff_pct]]
axes[1].errorbar([0], [diff_pct], yerr=yerr_lower,
                 fmt="o", color="#C44E52", capsize=10, capthick=2, markersize=10, linewidth=2.5)
axes[1].axhline(y=0, color="gray", linestyle="--", linewidth=1.2, label="无差异线 (0)")
axes[1].set_title("转化率差异 95%置信区间", fontsize=14, fontweight="bold", pad=10)
axes[1].set_ylabel("绝对差异 (%)", fontsize=12)
axes[1].set_xticks([0])
axes[1].set_xticklabels(["treatment - control"], fontsize=11)
axes[1].set_ylim(ci_low_pct - 0.3, ci_high_pct + 0.5)
axes[1].text(0.15, diff_pct,
             f"差异: {diff_pct:.2f}%\n95% CI: [{ci_low_pct:.2f}%, {ci_high_pct:.2f}%]\np = {p_value:.6f}",
             fontsize=10.5, va="center",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#F5F5F5", edgecolor="#CCCCCC"))
axes[1].legend(fontsize=10, loc="lower right")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/10_abtest_conversion.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.6 MDE 最小可检测效应

**样本量公式**：
$$
n = \frac{(Z_{\alpha/2} + Z_{\beta})^2 \cdot 2p(1-p)}{\Delta^2}
$$

**反推 MDE**：
$$
MDE = (Z_{\alpha/2} + Z_{\beta}) \cdot \sqrt{\frac{2p(1-p)}{n}}
$$


In [ ]:
z_alpha_2_val = 1.96
z_beta_val = 0.84
alpha = 0.05
beta = 0.20
power = 1 - beta
p_baseline = control_cr
n_per_group = min(n_control, n_treatment)

mde = (z_alpha_2_val + z_beta_val) * np.sqrt(2 * p_baseline * (1 - p_baseline) / n_per_group)
mde_relative = mde / p_baseline

print(f"显著性水平 α = {alpha}（Zα/2 = {z_alpha_2_val}）")
print(f"统计功效 1-β = {power}（Zβ = {z_beta_val}）")
print(f"基准转化率 p = {p_baseline*100:.4f}%")
print(f"每组样本量 n = {n_per_group:,}")
print(f"绝对 MDE = {mde*100:.4f}个百分点")
print(f"相对 MDE = {mde_relative*100:.2f}%")
print(f"实际相对lift = {rel_lift*100:.2f}%")
print(f"{'实际lift > 相对MDE，效应可被检测到。' if rel_lift > mde_relative else '实际lift < 相对MDE。'}")

In [ ]:
# 4.7 AA空转仿真
all_user_sorted = np.array(sorted(all_users_arr))
is_buyer_arr = np.array([1 if uid in buy_users_set else 0 for uid in all_user_sorted])
n_total_aa = len(all_user_sorted)
mid = n_total_aa // 2

n_aa_tests = 1000
aa_p_values = []
aa_significant_count = 0

for i in range(n_aa_tests):
    np.random.seed(i + 10000)
    perm = np.random.permutation(n_total_aa)
    c_idx = perm[:mid]
    t_idx = perm[mid:]
    x_c_a = is_buyer_arr[c_idx].sum()
    x_t_a = is_buyer_arr[t_idx].sum()
    n_c_a = len(c_idx)
    n_t_a = len(t_idx)
    p_pool_a = (x_c_a + x_t_a) / (n_c_a + n_t_a)
    se_a = np.sqrt(p_pool_a * (1 - p_pool_a) * (1/n_c_a + 1/n_t_a))
    z_a = (x_t_a/n_t_a - x_c_a/n_c_a) / se_a if se_a > 0 else 0
    p_a = 2 * (1 - stats.norm.cdf(abs(z_a)))
    aa_p_values.append(p_a)
    if p_a < 0.05:
        aa_significant_count += 1

type1_error_rate = aa_significant_count / n_aa_tests
print(f"AA测试次数：{n_aa_tests}")
print(f"p<0.05 的次数：{aa_significant_count}")
print(f"一类错误率（实验值）：{type1_error_rate:.4f}（理论值 α=0.05）")

In [ ]:
# 图11：AA测试 p值分布
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
n_bins, _, _ = axes[0].hist(aa_p_values, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].axvline(x=0.05, color="#C44E52", linestyle="--", linewidth=2, label=f"α = 0.05")
axes[0].set_title(f"AA测试 p值分布（{n_aa_tests}次）", fontsize=14, fontweight="bold", pad=10)
axes[0].set_xlabel("p值", fontsize=12)
axes[0].set_ylabel("频次", fontsize=12)
axes[0].legend(fontsize=11, loc="upper right")
axes[0].text(0.5, max(n_bins) * 0.92,
             f"一类错误率: {type1_error_rate:.4f}\n理论值 α = 0.05",
             ha="center", fontsize=11,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#F0F8FF", edgecolor="#4C72B0"))

sorted_p = np.sort(aa_p_values)
cdf = np.arange(1, len(sorted_p) + 1) / len(sorted_p)
axes[1].plot(sorted_p, cdf, color="#4C72B0", linewidth=2.5, label="实际分布")
axes[1].plot([0, 1], [0, 1], color="#C44E52", linestyle="--", linewidth=1.8, label="理想均匀分布")
axes[1].axvline(x=0.05, color="gray", linestyle=":", linewidth=1.2)
axes[1].axhline(y=0.05, color="gray", linestyle=":", linewidth=1.2)
axes[1].set_title("AA测试 p值累积分布（CDF）", fontsize=14, fontweight="bold", pad=10)
axes[1].set_xlabel("p值", fontsize=12)
axes[1].set_ylabel("累积比例", fontsize=12)
axes[1].legend(fontsize=11, loc="lower right")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/11_aa_test.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.8 业务决策结论

1. 仿真设定实验组获得约 2.98% 的相对转化提升
2. Z检验 p值 < 0.05，差异统计显著
3. 95%置信区间不包含 0，确认正向效应
4. 实际 lift > MDE，效应可被检测
5. AA测试一类错误率接近理论值 0.05，实验框架可靠
6. 决策建议：在仿真场景下，优惠券策略带来统计显著的转化提升。真实业务中需进一步验证 ROI、优惠券成本等。